# Actividad 5 – Marcas y Canales (Capítulo 5, Munzner 2014)
**Universidad Tecnológica de Panamá – Sistemas de Información, Control y Evaluación de Recursos Informáticos**

Este cuaderno resuelve las **Actividades 3 y 4** (Python). Los datos se leen directamente desde GitHub:
`https://github.com/Samycaballero/visualizacion-capitulo5-munzner`

> Ejecuta *Entorno → Ejecutar todas* (Runtime → Run all) en Colab.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

BASE = "https://raw.githubusercontent.com/Samycaballero/visualizacion-capitulo5-munzner/main/"
URL_TECHSTORE = BASE + "techstore_participacion_capitulo5.csv"
URL_PUNTOS    = BASE + "puntos_popout_capitulo5.csv"

plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})

---
## Actividad 3. Simulación del ranking de exactitud con matplotlib
**Objetivo:** comparar dos codificaciones de los mismos datos: **posición/longitud** (barras) vs. **área** (círculos).

In [ ]:
df_ts = pd.read_csv(URL_TECHSTORE)          # separador coma
df_ts

### 3.1 Barras – canal de posición/longitud

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(df_ts["Linea_Producto"], df_ts["Participacion_Pct"], color="#1F77B4")
ax.bar_label(bars, fmt="%d %%")
ax.set_ylabel("Participación (%)")
ax.set_title("TechStore Panamá – Participación por línea (posición/longitud)")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

### 3.2 Círculos – canal de área (sin ejes de referencia)

In [ ]:
K = 60                                          # constante: área (pts²) = valor * K
x = np.arange(len(df_ts))
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.scatter(x, np.zeros(len(x)), s=df_ts["Participacion_Pct"] * K, color="#1F77B4", alpha=0.85)
for xi, nombre in zip(x, df_ts["Linea_Producto"]):
    ax.annotate(nombre, (xi, 0), xytext=(0, -38), textcoords="offset points", ha="center", fontsize=9)
ax.set_xlim(-0.7, len(x) - 0.3)
ax.set_ylim(-1, 1)
ax.axis("off")                                  # sin ejes de referencia
ax.set_title("Mismos datos codificados con área de círculos")
plt.tight_layout()
plt.show()

### 3.3 Conclusión (Ley de Stevens)
La percepción de la magnitud de un estímulo sigue una **ley de potencia** `S = k · I^n`. Para la **longitud** el exponente es
casi **1** (n ≈ 1.0): la longitud percibida es prácticamente proporcional a la real, por lo que una barra el doble de larga se ve
el doble de larga. Para el **área** el exponente es **menor que 1** (n ≈ 0.7): el área se percibe *comprimida* y se subestiman
las diferencias entre círculos.

En los datos de TechStore, Accesorios (18 %) y Componentes PC (14 %) difieren en 4 puntos: con barras la diferencia se aprecia de
inmediato porque además las barras comparten una **escala común alineada** (el canal más efectivo del ranking de Munzner y el más
exacto según Cleveland y McGill), mientras que con círculos sin ejes los dos tamaños se ven casi idénticos y hay que "adivinar".

**Conclusión:** el gráfico de barras (posición/longitud) permite comparar los valores con mayor exactitud que los círculos (área),
porque el sistema perceptual juzga longitudes de forma casi lineal y relativa a una referencia común (Ley de Weber), mientras que
el área se subestima y no tiene línea base.

---
## Actividad 4. Popout y agrupación con seaborn/matplotlib
**Objetivo:** observar el fenómeno de *popout* (procesamiento preatencional) y la agrupación perceptual por color.

In [ ]:
df_p = pd.read_csv(URL_PUNTOS, sep=";", encoding="utf-8")   # separador punto y coma
# El archivo trae "Si"/"No" (en Excel aparece "Sí"): se normaliza para aceptar ambas formas
df_p["Es_Atipico"] = df_p["Es_Atipico"].str.strip().str.replace("Sí", "Si")
print(df_p.shape)
print(df_p["Grupo"].value_counts().to_string())
df_p.head()

### 4.1 Versión A – todos los puntos con el mismo color y forma

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 5))
ax.scatter(df_p["X"], df_p["Y"], color="#1F77B4", marker="o", s=45)
ax.set_xlabel("X"); ax.set_ylabel("Y")
ax.set_title("A. Todos los puntos iguales")
plt.tight_layout(); plt.show()

### 4.2 Versión B – atípicos resaltados con color y forma distintos

In [ ]:
atipicos = df_p[df_p["Es_Atipico"] == "Si"]
normales = df_p[df_p["Es_Atipico"] != "Si"]
print("Puntos atípicos:\n", atipicos[["X", "Y", "Grupo"]].to_string(index=False))

fig, ax = plt.subplots(figsize=(6.5, 5))
ax.scatter(normales["X"], normales["Y"], color="#1F77B4", marker="o", s=45, label="Normales")
ax.scatter(atipicos["X"], atipicos["Y"], color="red", marker="X", s=150, label="Atípicos")
ax.set_xlabel("X"); ax.set_ylabel("Y")
ax.set_title("B. Atípicos resaltados (popout)")
ax.legend()
plt.tight_layout(); plt.show()

**Comparación:** en la versión **A** hay que *buscar* los dos atípicos entre 50 puntos idénticos (búsqueda serial: el tiempo crece con
el número de distractores). En la versión **B** los atípicos **saltan a la vista de inmediato** (*popout*): al diferir del resto en un
canal individual —el **color** (rojo vs. azul), reforzado por la **forma** (X vs. círculo)— el sistema visual los detecta de forma
preatencional, con un tiempo de detección que no depende del número de distractores. Por tanto, **la versión B es la que aprovecha el
fenómeno de popout** descrito en el capítulo.

### 4.3 Versión C – agrupación perceptual por color (`seaborn.scatterplot(hue='Grupo')`)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
sns.scatterplot(data=df_p, x="X", y="Y", hue="Grupo", palette=["#1F77B4", "#FF7F0E", "#2CA02C"], s=60, ax=ax)
ax.set_title("C. Agrupación perceptual por color (Grupo)")
ax.legend(title="Grupo")
plt.tight_layout(); plt.show()

**Explicación:** el **matiz (color)** es un canal de *identidad* (responde a "qué es"): los puntos que comparten el mismo color se
perciben automáticamente como un **conjunto** (agrupación por similitud en un canal de identidad). Esto se refuerza con la **proximidad
espacial**: Grupo 1 (azul) forma una nube abajo-izquierda, Grupo 2 (naranja) en el centro-derecha y Grupo 3 (verde) arriba. El color
permite distinguir y recordar cada categoría sin leer ninguna etiqueta; además se observa que los atípicos pertenecen a Grupo 1 (15; 15)
y Grupo 3 (0,2; 14,5) pero quedan lejos de su nube.

> Según Munzner, la agrupación por color es más débil que la contención o la conexión, pero más práctica cuando hay pocas categorías
(≤ 6–12 matices distinguibles).

---
### Bibliografía
Munzner, T. (2014). *Visualization Analysis and Design*. Boca Raton, FL: CRC Press, A K Peters Visualization Series.